# Week 61, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w061d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class M:
    def __init__(self, n):
        self.n = n
    def __enter__(self):
        print("in", self.n)
        return self.n * 10
    def __exit__(self, t, e, tb):
        print("out", self.n, t.__name__ if t else "")
        return t is KeyError

with M(1) as a, M(2) as b:
    print(a + b)
    raise KeyError
print("after")

**Predict before you run:** what will this print?



In [ ]:
from contextlib import contextmanager

@contextmanager
def tag(name):
    print(f"<{name}>")
    yield
    print(f"</{name}>")

try:
    with tag("p"):
        raise ValueError
except ValueError:
    print("caught")

**Predict before you run:** what will this print?



In [ ]:
from contextlib import ExitStack
out = []
with ExitStack() as s:
    for i in range(3):
        s.callback(out.append, i)
print(out)

**Predict before you run:** what will this print?



In [ ]:
from contextlib import suppress, nullcontext
with suppress(IndexError):
    [][0]
with nullcontext("value") as v:
    print(v)

**Predict before you run:** what will this print?



In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    inside = os.path.isdir(d)
print(inside, os.path.isdir(d))

### Exercise: Retrying a block

Write `attempts(times, exceptions)`: a generator yielding a context manager for each attempt, so that callers write `for attempt in attempts(3, (ConnectionError,)): with attempt: do_something()`. Each attempt's manager suppresses the listed exceptions **unless** it's the last attempt; after an attempt whose block succeeds, the generator stops (so the loop ends). Each manager exposes `number` (from 1).

In [ ]:
from contextlib import contextmanager

def attempts(times, exceptions):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_attempts():\n    \"generators and context managers\"\n    calls = []\n    replies = iter([ConnectionError(\"a\"), ConnectionError(\"b\"), \"ok\"])\n    for attempt in attempts(5, (ConnectionError,)):\n        with attempt:\n            calls.append(attempt.number)\n            r = next(replies)\n            if isinstance(r, Exception):\n                raise r\n    assert calls == [1, 2, 3]\n    tries = []\n    try:\n        for attempt in attempts(2, (ConnectionError,)):\n            with attempt:\n                tries.append(attempt.number)\n                raise ConnectionError(\"down\")\n        assert False\n    except ConnectionError:\n        pass\n    assert tries == [1, 2]\n    try:\n        for attempt in attempts(3, (ConnectionError,)):\n            with attempt:\n                raise KeyError\n        assert False\n    except KeyError:\n        pass")

### Exercise: Splitting a file into parts

Write `split_lines(lines, size, prefix)`: write the lines into files named `f"{prefix}{n}.txt"` (n from 1), `size` lines per file, using `itertools.batched` and **one** `ExitStack` that keeps every part file open until all are written; return the list of file names. If writing fails part-way, every file opened must still be closed.

In [ ]:
from contextlib import ExitStack
from itertools import batched

def split_lines(lines, size, prefix):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_split():\n    \"ExitStack with itertools\"\n    names = split_lines([\"a\", \"b\", \"c\", \"d\", \"e\"], 2, \"part\")\n    assert names == [\"part1.txt\", \"part2.txt\", \"part3.txt\"]\n    with open(\"part3.txt\") as f:\n        assert f.read() == \"e\\n\"\n    with open(\"part1.txt\") as f:\n        assert f.read() == \"a\\nb\\n\"\n    assert split_lines([], 3, \"none\") == []\n    assert \"ExitStack(\" in source and \"batched(\" in source")

### Exercise: Forge: a job runner

Write `JobRunner` (with its context and result classes) to the specification above.

In [ ]:
import io, os, tempfile, time
from contextlib import ExitStack, redirect_stdout

class JobRunner:
    def __init__(self, workdir, clock=time.perf_counter):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\n\ndef make(folder=\"runs\"):\n    ticks = iter(range(0, 1000, 2))\n    runner = JobRunner(folder, clock=lambda: next(ticks))\n    return runner\n\ndef test_register():\n    \"registering jobs\"\n    runner = make()\n    @runner.job\n    def hello(ctx):\n        return \"hi\"\n    @runner.job(name=\"report\")\n    def make_report(ctx, title=\"T\"):\n        return title\n    assert runner.jobs() == [\"hello\", \"report\"] and hello(None) == \"hi\"\n    try:\n        runner.run(\"missing\")\n        assert False\n    except KeyError:\n        pass\n\ndef test_success():\n    \"a job that works\"\n    runner = make(\"runs1\")\n    seen = {}\n    @runner.job\n    def count_words(ctx, text):\n        with ctx.open(\"draft.txt\", \"w\") as f:\n            f.write(text)\n        seen[\"scratch\"] = ctx.scratch\n        f2 = ctx.open(\"draft.txt\")\n        words = len(f2.read().split())\n        seen[\"file\"] = f2\n        print(\"counted\", words)\n        ctx.publish(\"count.txt\", str(words))\n        ctx.on_cleanup(seen.setdefault, \"cleaned\", True)\n        return words\n    r = runner.run(\"count_words\", text=\"the cat sat\")\n    assert (r.status, r.value, r.error, r.log, r.seconds) == (\"ok\", 3, None, [\"counted 3\"], 2)\n    assert not os.path.exists(seen[\"scratch\"]) and seen[\"file\"].closed and seen[\"cleaned\"]\n    with open(os.path.join(\"runs1\", \"outputs\", \"count.txt\")) as f:\n        assert f.read() == \"3\"\n    assert runner.history == [r]\n\ndef test_failure():\n    \"a job that fails\"\n    runner = make(\"runs2\")\n    seen = {}\n    @runner.job\n    def broken(ctx):\n        seen[\"scratch\"] = ctx.scratch\n        seen[\"file\"] = ctx.open(\"notes.txt\", \"w\")\n        ctx.publish(\"half.txt\", \"partial\")\n        print(\"about to fail\")\n        raise ValueError(\"bad data\")\n    r = runner.run(\"broken\")\n    assert (r.status, r.value, r.error, r.log) == (\"failed\", None, \"ValueError\", [\"about to fail\"])\n    assert not os.path.exists(seen[\"scratch\"]) and seen[\"file\"].closed\n    assert not os.path.exists(os.path.join(\"runs2\", \"outputs\", \"half.txt\"))\n    @runner.job\n    def fine(ctx):\n        ctx.publish(\"a.txt\", \"A\")\n        return 1\n    assert runner.run(\"fine\").status == \"ok\" and [x.status for x in runner.history] == [\"failed\", \"ok\"]\n    assert os.listdir(\"runs2\") == [\"outputs\"]")

> **Where the guarantees come from.** The runner promises four things: files closed, scratch removed, output captured, failures rolled back. None of them is written as cleanup code in the job; each comes from a context manager wrapped around it. The ExitStack closes whatever the job opened, the TemporaryDirectory removes the folder, redirect_stdout restores printing, and the rollback runs in the except clause before the temporary folder goes. Because every guarantee lives in a with statement, it holds on every path, including the ones nobody thought to test. That's the lesson of the week: put cleanup where the language guarantees it runs.

### Exercise: Forge bonus: layered settings

Programs often need settings that tests or parts of a program change temporarily. Write `Settings(**defaults)`:

- `get(key)` and `settings[key]`: the current value (`KeyError` for a key that has no default).
- `override(**changes)`: a context manager (that also works as a **decorator**, via `ContextDecorator`) pushing a layer of changes for the block, and popping it at the end, even after an error. Layers nest; the newest wins. Entering raises `KeyError` for an unknown key, and `RuntimeError("settings are frozen")` while frozen. Entering returns the settings.
- `frozen()`: a context manager (reentrant) during which no override can be entered.
- `snapshot()`: a dict of every key's current value; `depth()`: how many layers are active.

In [ ]:
from contextlib import contextmanager, ContextDecorator

class Settings:
    def __init__(self, **defaults):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_layers():\n    \"nested overrides\"\n    s = Settings(colour=\"blue\", size=10, debug=False)\n    assert s.get(\"colour\") == \"blue\" and s[\"size\"] == 10\n    with s.override(colour=\"red\") as same:\n        assert same is s and s[\"colour\"] == \"red\" and s.depth() == 1\n        with s.override(size=12, colour=\"green\"):\n            assert s.snapshot() == {\"colour\": \"green\", \"size\": 12, \"debug\": False}\n        assert s.snapshot() == {\"colour\": \"red\", \"size\": 10, \"debug\": False}\n    assert s.depth() == 0 and s[\"colour\"] == \"blue\"\n    try:\n        with s.override(debug=True):\n            raise ValueError\n    except ValueError:\n        pass\n    assert s[\"debug\"] is False\n    try:\n        with s.override(font=\"mono\"):\n            pass\n        assert False\n    except KeyError:\n        pass\n    assert s.depth() == 0\n\ndef test_decorator_and_frozen():\n    \"decorators and freezing\"\n    s = Settings(debug=False, level=1)\n    @s.override(debug=True)\n    def check():\n        return s[\"debug\"]\n    assert check() is True and s[\"debug\"] is False and check() is True\n    with s.frozen():\n        try:\n            with s.override(level=2):\n                pass\n            assert False\n        except RuntimeError as e:\n            assert str(e) == \"settings are frozen\"\n        with s.frozen():\n            pass\n        try:\n            s.override(level=3).__enter__()\n            assert False\n        except RuntimeError:\n            pass\n    with s.override(level=5):\n        assert s[\"level\"] == 5")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [with and why it exists](w061d1.html), [context manager classes](w061d2.html), [contextlib](w061d3.html), [ExitStack](w061d4.html) or [managing resources safely](w061d5.html).

## Mixed review

## Forge: a job runner

A job runner runs named jobs and promises to clean up after each one, however it ends. Write `JobRunner(workdir, clock=time.perf_counter)`:

- `@runner.job` or `@runner.job(name=...)` registers a function (returned unchanged); `runner.jobs()` lists names, sorted.
- `runner.run(name, **params)` (`KeyError` for an unknown job) calls `job(ctx, **params)` with a fresh **context** and returns a **result**, also appended to `runner.history`.
- Each run gets a new temporary folder **inside** `workdir` (create `workdir` if needed; `tempfile.TemporaryDirectory(dir=workdir)`), removed at the end of the run.
- The context offers: `ctx.scratch` (that folder's path); `ctx.path(name)`; `ctx.open(name, mode="r")`, opening a file in the scratch folder that's closed automatically at the end of the run; `ctx.on_cleanup(func, *args)`, to register cleanups; and `ctx.publish(name, text)`, writing `workdir/outputs/name` **atomically** (a `.tmp` file, then `os.replace`) and returning its path.
- Everything the job **prints** is captured.
- If the job raises an `Exception`, the run doesn't raise: it's recorded as failed, and every file the job published during that run is **removed**.
- The result has `name`, `status` (`"ok"` or `"failed"`), `value` (the job's return value, or `None`), `error` (the exception's type name, or `None`), `log` (the captured lines) and `seconds` (by the clock).

## Forge bonus: layered settings

## Weekly quiz

## Week 61 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain exactly what with does with __enter__ and __exit__, and the order of several managers. ([Day 1](w061d1.html))
- I can write context manager classes for timing, temporary changes, suppression, transactions and nesting. ([Day 2](w061d2.html))
- I can write managers with @contextmanager safely and use contextlib's helpers. ([Day 3](w061d3.html))
- I can use ExitStack for run-time numbers of resources, undo steps, and hand resources on with pop_all. ([Day 4](w061d4.html))
- I can manage temporary space, idempotent closing, pools, deadlines and leak tests. ([Day 5](w061d5.html))
- I built a job runner and layered settings. (Today)

Tomorrow is a **rest day**. Next week: **The memory model**: names, references, identity and garbage collection.